
# Demo 2 - Predict



In [ ]:
# Dependencies:

# !pip install openrouter
# !pip install dspy-ai python-dotenv
# !pip install litellm
# !pip install wikipedia-api

# Part 1 - Predict with DSPy 

## 1. Configure a model

In [ ]:
import os
import dspy
from dotenv import load_dotenv

load_dotenv()

lm = dspy.LM(
    model= "anthropic/claude-haiku-4.5", 
    api_key=os.getenv("OPENROUTER_API_KEY"),
    api_base="https://openrouter.ai/api",
    extra_headers={
        "HTTP-Referer": "https://my-app.com",
        "X-Title": "My App Name"
    }
)
dspy.configure(lm=lm)

## 2. Call the LM for a single-pass prediction

In [3]:
qa = dspy.Predict('question -> answer')
response = qa(question="What is DSPy?")
print(response.answer)

DSPy is a framework for algorithmically optimizing language model (LM) prompts and weights. It stands for "Declarative Self-improving Language Programs" and was developed by researchers at Stanford University.

Key characteristics of DSPy include:

1. **Prompt Optimization**: It automatically optimizes prompts rather than requiring manual prompt engineering.

2. **Modular Architecture**: DSPy programs are composed of modules that call language models, allowing for structured and composable LM-based applications.

3. **Self-Improving**: The framework can learn and improve its own prompts and parameters based on training data and evaluation metrics.

4. **Abstraction Layer**: It provides an abstraction layer over language models, making it easier to build complex LM applications without worrying about low-level prompt details.

5. **Framework Independence**: DSPy can work with various language models and backends.

The main goal of DSPy is to move away from ad-hoc prompt engineering towa

## 3. Define a Signature for the prediction input/output

In [4]:
from typing import Literal

Season = Literal[
    "spring", "summer", "autumn", "winter",
]

class HaikuBot(dspy.Signature):
    """
    Write a classical haiku given the provided inputs.
    """
    location: str = dspy.InputField()
    mood: str = dspy.InputField()
    season: Season = dspy.InputField()
    haiku: str = dspy.OutputField()

## 4. Use the signature for a single-pass prediction (DSPy)

In [5]:
haiku_bot = dspy.Predict(HaikuBot)
result = haiku_bot(location="Bodega Bay", mood="mysterious", season="autumn")
print(result.haiku)

Fog rolls through the bay
Secrets hide in autumn mist
Waves whisper unknown


## 4.1. Use the signature for a single-pass prediction (LangChain)

In [4]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "auto",
    model_provider="openrouter",
)

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie)
response = model_with_structure.invoke("Provide details about the movie Inception")
print(response)  # Movie(title="Inception", year=2010, director="Christopher Nolan", rating=8.8)

title='Inception' year=2010 director='Christopher Nolan' rating=8.8


# 5. Simple LangChain invocation with system prompt 

In [ ]:
# !pip install -qU langchain langchain-openrouter

Using model

In [ ]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

conversation = [
    SystemMessage("You are a helpful assistant that translates English to French."),
    HumanMessage("Translate: I love programming."),
    AIMessage("J'adore la programmation."),
    HumanMessage("Translate: I love building applications.")
]

response = model.invoke(conversation)
print(response)  # AIMessage("J'adore créer des applications.")

Using agent

In [3]:
# pip install -qU langchain langchain-openrouter
from langchain.agents import create_agent

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

agent = create_agent(
    model="openrouter:anthropic/claude-sonnet-4-6",
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in San Francisco?"}]}
)
print(result["messages"][-1].content_blocks)

[{'type': 'text', 'text': 'The weather in San Francisco is sunny! It looks like a great day to get outside and enjoy the city. ☀️'}]


## 6. Chain of thought

In [13]:
cot_haiku_bot = dspy.ChainOfThought(HaikuBot)
result = cot_haiku_bot(location="Bodega Bay", mood="mysterious", season="autumn")
print(result.haiku)
print("reasoning")
print(result.reasoning)

Fog veils the coastline
Waves whisper ancient secrets
Shadows deepen here
reasoning
Bodega Bay in autumn evokes a mysterious atmosphere with its rugged California coastline. The season brings darker, moodier weather—gray skies, crashing waves, and fog rolling in from the Pacific. The combination of the isolated bay setting with autumn's inherent melancholy creates an ideal environment for a haiku capturing mystery. I'll focus on sensory imagery: the sound of waves, the visual of mist, and the feeling of solitude that permeates the location during this season.


In LangChain Reasoning, is available as part of the content blocks, if the model supports it. 

In [5]:
response = model.invoke("Why do parrots have colorful feathers?")
reasoning_steps = [b for b in response.content_blocks if b["type"] == "reasoning"]
print(" ".join(step["reasoning"] for step in reasoning_steps))

**Explaining coloration reasons**

I need to provide a simple answer about why animals have color. I'll explain things like pigments and structural coloration, which serve functions such as camouflage, mate attraction, species recognition, and social signaling. I should also mention the role of UV light and habitat in this context. It’s worth noting that not all colors are bright to attract predators, to keep my response concise and informative.


## 7. Program Of Thought


In [14]:
# import dspy
# sandbox = dspy.LocalSandbox()
# expr = "value = 2*5 + 4\nvalue"
# answer = sandbox.execute(expr)
# answer
import os
import dspy
from dotenv import load_dotenv

load_dotenv()

pot_llm = dspy.LM("anthropic/claude-haiku-4.5", 
                      api_key=os.getenv("OPENROUTER_API_KEY"),
                      api_base="https://openrouter.ai/api",
                        )

dspy.configure(lm=pot_llm)

class BasicGenerateAnswer(dspy.Signature):
    question = dspy.InputField()
    answer = dspy.OutputField()

pot = dspy.ProgramOfThought(BasicGenerateAnswer)
problem = "2*5 + 4"
pot(question=problem).answer

'14'

## Also see

- https://openrouter.ai/docs/quickstart
- https://openrouter.ai/models
- https://dspy.ai/getting-started/installation/
- [Composing modules](https://dspy.ai/getting-started/composing-modules/)